# MLflow & database

Тестирование подключения к MLFLow и БД из Jupyter Notebook

## Configuration

In [1]:
# Хак, чтобы добавить корень проекта в path для импорта модулей
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (pyproject.toml). Open this notebook from the repo.")

sys.path.append(str(find_repo_root()))
sys.path

['/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python312.zip',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor/.venv/lib/python3.12/site-packages',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor']

In [2]:
# Код для загрузки конфига и подключения к MLFlow
# Для работы с продом в корне проекта должен быть .env файл с всеми нужными переменными окружения
# Для локальной разработки .env не нужен
from jupyter_utils import setup_jupyter_notebook

# Подключение к проду
setup_jupyter_notebook(environment='prod', experiment='test')

# Подключение к локальным БД и MLFlow
# setup_jupyter_notebook(environment='local', experiment='test')

Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: test
Database: localhost:15432/stocks_advisor_db


## MLFlow

Пример логгирования эксперимента в MLFlow

In [3]:
from datetime import UTC, datetime
import mlflow

run_name = f"smoke-test-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("source", "mlflow_smoke_test")
    mlflow.log_metric("ping", 1.0)
    mlflow.log_dict({"status": "ok"}, "smoke.json")

print(f"Run ID: {run.info.run_id}")
print(f"Run name: {run_name}")
print("Smoke test passed.")

🏃 View run smoke-test-20260530-095125 at: http://localhost:5050/#/experiments/2/runs/8adffad47cb64e9588a1d8404248f8b2
🧪 View experiment at: http://localhost:5050/#/experiments/2
Run ID: 8adffad47cb64e9588a1d8404248f8b2
Run name: smoke-test-20260530-095125
Smoke test passed.


## Database queries

Примеры загрузки данных из PostgreSQL:

- Стоимость акций
- Данные по новостям: тикеры, сектор, sentiment

In [4]:
# Загрузка свечей MOEX

from app.core.database import AssetCandleRepository, get_db_session

TICKER = 'SBER'
LIMIT = 100

async with get_db_session() as session:
    repo = AssetCandleRepository(session)
    stocks_df = await repo.get_dataframe_by_ticker(TICKER, limit=LIMIT)

print(f'{TICKER}: {len(stocks_df)} rows')
stocks_df.tail()

SBER: 100 rows


,begin,open,high,low,close,volume,value,ticker
95,2026-05-23 14:00:00,322.79,322.80,322.60,322.75,98703.0,31854933.66,SBER
96,2026-05-23 15:00:00,322.75,322.80,322.60,322.64,65450.0,21122415.55,SBER
97,2026-05-23 16:00:00,322.64,322.79,322.57,322.65,29537.0,9531350.86,SBER
98,2026-05-23 17:00:00,322.65,322.75,322.51,322.65,64055.0,20665351.05,SBER
99,2026-05-23 18:00:00,322.64,322.77,322.60,322.72,52276.0,16868518.97,SBER


In [5]:
# Загрузка данных по новостям

from app.core.database import NewsArticleRepository, get_db_session

LIMIT = 100

async with get_db_session() as session:
    repo = NewsArticleRepository(session)
    enrichments_df = await repo.get_all_enrichments_as_dataframe(limit=LIMIT, ticker=TICKER, sector='MOEXFN')

print(f'Enrichments: {len(enrichments_df)} rows')
enrichments_df.head()

Enrichments: 100 rows


,id,news_article_id,published_at,tickers,sector,sentiment,sentiment_score
0,49186,134601,2024-11-25 18:36:00,[SBER],MOEXFN,negative,0.937763
1,49191,134598,2024-11-25 19:05:00,"[VTBR, ROSN, LKOH, GAZP, SBER, TATN]",MOEXFN,negative,0.872153
2,49210,134585,2024-11-25 21:54:00,"[SBER, VTBR, TATN]",MOEXFN,negative,0.540392
3,49298,135456,2024-11-26 10:16:00,"[SBER, VTBR, TATN, LKOH]",MOEXFN,negative,0.928977
4,49317,135441,2024-11-26 12:39:00,[SBER],MOEXFN,neutral,0.577350
